In [0]:
# ==========================================================
# 06_feature_importance_export.ipynb
# Fully Correct Version (Pipeline Fix)
# ==========================================================

import pandas as pd
import mlflow
import os

# -------------------------------------------------
# Required Serverless Fix
# -------------------------------------------------

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlflow_tmp"

# -------------------------------------------------
# Load Model
# -------------------------------------------------

run_id = "e4dc997b26944885abc717b5dd23da8e"

model_uri = f"runs:/{run_id}/best_model"

rf_pipeline = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
)

# Extract actual RF model from pipeline
rf_model = rf_pipeline.stages[-1]

print("Loaded RF model successfully")

# -------------------------------------------------
# Extract Feature Importances
# -------------------------------------------------

importances = rf_model.featureImportances.toArray()

feature_names = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID_idx",
    "DOLocationID_idx",
    "payment_type_idx",
    "trip_duration",
    "pickup_hour",
    "is_weekend"
]

fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

spark.createDataFrame(fi_df) \
    .write.mode("overwrite") \
    .parquet("/Volumes/workspace/default/taxi_data/feature_importance")

print("Feature importance exported successfully.")

In [0]:
spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/feature_importance"
).orderBy("importance", ascending=False).show()

In [0]:
# mOdle Summary
from pyspark.sql import Row

model_metrics = [
    Row(model="Linear Regression", rmse=12.56),
    Row(model="Decision Tree", rmse=14.97),
    Row(model="Random Forest", rmse=4.52),
    Row(model="Random Forest Tuned", rmse=4.92),
    Row(model="Gradient Boosted Trees", rmse=3.53)
]

spark.createDataFrame(model_metrics) \
    .write.mode("overwrite") \
    .parquet("/Volumes/workspace/default/taxi_data/model_performance")

In [0]:
run_id = "ced84abe41a74eb1abc7ef5865e5f68a"

model_uri = f"runs:/{run_id}/model"

gbt_pipeline = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp"
)

# Extract actual model if wrapped in pipeline
if hasattr(gbt_pipeline, "stages"):
    gbt_model = gbt_pipeline.stages[-1]
else:
    gbt_model = gbt_pipeline

print("GBT model loaded successfully")

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler

df = spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/silver_2019"
)

feature_cols = [
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID_idx",
    "DOLocationID_idx",
    "payment_type_idx",
    "trip_duration",
    "pickup_hour",
    "is_weekend"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data_ml = assembler.transform(df) \
                   .select(
                       "features",
                       col("fare_amount").alias("label"),
                       "pickup_month"
                   )

test = data_ml.filter(col("pickup_month") > 9)

print("Test set rebuilt:", test.count())

final_predictions = gbt_model.transform(test) \
    .select("label", "prediction", "pickup_month")

final_predictions.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/taxi_data/final_predictions"
)

print("Final predictions exported successfully.")

In [0]:
spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/final_predictions"
).sample(0.05).coalesce(1).write.mode("overwrite") \
 .option("header", "true") \
 .csv("/Volumes/workspace/default/taxi_data/export_predictions")

In [0]:
# Model performance
spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/model_performance"
).coalesce(1).write.mode("overwrite").option("header","true") \
 .csv("/Volumes/workspace/default/taxi_data/export_model_performance")

# Feature importance
spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/feature_importance"
).coalesce(1).write.mode("overwrite").option("header","true") \
 .csv("/Volumes/workspace/default/taxi_data/export_feature_importance")

# Strong scaling
spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/strong_scaling"
).coalesce(1).write.mode("overwrite").option("header","true") \
 .csv("/Volumes/workspace/default/taxi_data/export_strong_scaling")

# Weak scaling
spark.read.parquet(
    "/Volumes/workspace/default/taxi_data/weak_scaling"
).coalesce(1).write.mode("overwrite").option("header","true") \
 .csv("/Volumes/workspace/default/taxi_data/export_weak_scaling")